In [ ]:

# KD-Tree vs kNN Brute: kNN speed + accuracy benchmark
# Builds and times KD-Tree vs Brute for p=2 (Euclidean) 
# and p=1 (Manhattan) with the same k=9

import time
from time import perf_counter
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
)

#  Load preprocessed, scaled data
X_train = pd.read_csv("X_train_scaled.csv")
X_test  = pd.read_csv("X_test_scaled.csv")
y_train = pd.read_csv("y_train.csv").squeeze()
y_test  = pd.read_csv("y_test.csv").squeeze()


def benchmark_knn(algorithm, p, k=9, name=None, plot_cm=False):
    """
    algorithm: 'kd_tree' or 'brute'
    p: 1 (Manhattan) or 2 (Euclidean)
    k: neighbors
    """
    label = name or f"{algorithm.upper()}-p{p}"

    # build (fit) time
    t0 = perf_counter()
    knn = KNeighborsClassifier(
        n_neighbors=k, algorithm=algorithm, p=p, weights="uniform"
    )
    knn.fit(X_train, y_train)
    build_s = perf_counter() - t0

    # pure neighbor-search time (separate)
    t1 = perf_counter()
    _ = knn.kneighbors(X_test, n_neighbors=k, return_distance=False)
    kneigh_s = perf_counter() - t1

    # full predict time (includes neighbor search + voting)
    t2 = perf_counter()
    y_pred = knn.predict(X_test)
    predict_s = perf_counter() - t2

    # metrics
    y_proba = knn.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    if plot_cm:
        cm = confusion_matrix(y_test, y_pred)
        ConfusionMatrixDisplay(confusion_matrix=cm).plot(values_format="d")
        plt.title(f"{label} - Confusion Matrix")
        plt.tight_layout()
        plt.show()

    return {
        "Setup": label,
        "Algorithm": algorithm,
        "p": p,
        "k": k,
        "Build (s)": build_s,
        "kNeighbors (s)": kneigh_s,
        "Predict (s)": predict_s,
        "Accuracy": acc,
        "ROC-AUC": auc,
    }

#  Run benchmarks 
rows = []
rows.append(benchmark_knn("kd_tree", p=2, k=9, name="KD-Tree Euclidean", plot_cm=False))
rows.append(benchmark_knn("kd_tree", p=1, k=9, name="KD-Tree Manhattan", plot_cm=False))
rows.append(benchmark_knn("brute",   p=2, k=9, name="Brute Euclidean",  plot_cm=False))
rows.append(benchmark_knn("brute",   p=1, k=9, name="Brute Manhattan",  plot_cm=False))

results = pd.DataFrame(rows).set_index("Setup")
print("=== Benchmark Results ===")
print(results[["Algorithm","p","k","Build (s)","kNeighbors (s)","Predict (s)","Accuracy","ROC-AUC"]])

# save to csv
results.to_csv("knn_kdtree_benchmark.csv")
print("\nSaved: knn_kdtree_benchmark.csv")

# Quick visuals: neighbor-search time and accuracy
ax = results[["kNeighbors (s)"]].plot(kind="bar", figsize=(7,4), title="kNN Neighbor-Search Time (lower is better)")
plt.ylabel("Seconds"); plt.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.show()

ax = results[["Accuracy","ROC-AUC"]].plot(kind="bar", figsize=(7,4), title="kNN Accuracy & AUC by Setup")
plt.ylim(0,1); plt.ylabel("Score"); plt.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.show()
